#### Thử nghiệm: Ném HẾT feature vào train + đánh trọng số ưu tiên feature quan trọng

So sánh 3 cách trên cùng target `base_price`, cùng test-set:

- **(A) Baseline hiện tại** — 10 feature đã chọn lọc (`B_NUM` trong `_common_train.py`)
- **(B) Ném HẾT ~46 cột** khả dụng (bỏ ID/timestamp/target rò rỉ) — không đánh trọng số, để cây tự lọc
- **(C) Ném HẾT + đánh trọng số** ưu tiên feature quan trọng (dùng `feature_weights` của XGBoost —
  tham số chính thức làm tăng xác suất feature đó được chọn khi cây tách nhánh, dựa trên permutation
  importance đã đo được trước đó: `quote_distance`/`quote_duration` quan trọng nhất)

⚠️ Cây quyết định **không nhân trọng số trực tiếp vào input** như neural network — `feature_weights`
chỉ ảnh hưởng đến **xác suất được lấy mẫu** khi xây mỗi cây (qua `colsample_bytree`/`colsample_bynode`),
không phải nhân thẳng vào giá trị.

**1. Nạp toàn bộ cột khả dụng (loại bỏ ID/timestamp thô/target rò rỉ)**

In [1]:
import warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

# CAM dung: id, timestamp thô, target roi ri, split/evaluation_month (dung de loc, khong phai feature)
CAT_ALL = ["service_name", "pickup_location_name", "dropoff_location_name", "pickup_hex_id_7",
           "weather_main", "weather_description"]
NUM_ALL = [
    "pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude",
    "quote_distance", "quote_duration",
    "pricing_quote_count_5m_lag", "pricing_avg_shown_multiplier_5m_lag",
    "pricing_demand_index_5m_lag", "pricing_supply_index_5m_lag", "pricing_market_imbalance_5m_lag",
    "weather_temp", "weather_feels_like", "weather_pressure", "weather_humidity",
    "weather_wind_speed", "weather_wind_deg_sin", "weather_wind_deg_cos", "weather_clouds_all",
    "weather_age_minutes", "weather_missing", "weather_is_clear", "weather_is_clouds",
    "weather_is_rain", "weather_is_mist", "weather_is_thunderstorm", "weather_is_drizzle",
    "requested_lag_minutes", "actual_observation_age_minutes",
    "latest_observed_price", "latest_observed_multiplier",
    "latest_observed_quote_distance", "latest_observed_quote_duration",
    "history_60m_observation_count", "history_60m_price_mean", "history_60m_price_std",
    "history_60m_price_min", "history_60m_price_max", "history_60m_multiplier_mean",
    "history_60m_price_slope_per_minute", "gio_vn", "thu_vn", "target_is_weekend",
]
B_NUM_CU = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_base",
            "history_60m_price_mean", "history_60m_price_std", "history_60m_price_slope_per_minute",
            "latest_observed_quote_distance", "latest_observed_quote_duration", "actual_observation_age_minutes"]
CAT_CU = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay chuan_bi_du_lieu.ipynb truoc!"
COLS = list(dict.fromkeys(CAT_ALL + NUM_ALL + CAT_CU + B_NUM_CU +
    ["target_shown_price", "target_shown_multiplier", "latest_observed_multiplier",
     "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]  # cot tu tinh, khong co san trong parquet
df = pd.read_parquet(PREP, columns=COLS)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)
print(f"Nap {len(df):,} dong | (A) baseline: {len(CAT_CU)+len(B_NUM_CU)} feature "
      f"| (B)/(C) nem het: {len(CAT_ALL)+len(NUM_ALL)} feature")

Nap 6,897,051 dong | (A) baseline: 14 feature | (B)/(C) nem het: 49 feature


**2. Đánh trọng số cho (C) — dựa trên permutation importance đã đo trước đó**

`quote_distance`/`quote_duration` chiếm ~92% importance đã đo được → đánh trọng số cao (10x) cho 2
cột này, trọng số vừa (3x) cho nhóm feature "biết là có liên quan" (dịch vụ, giá quan sát gần nhất,
lịch sử giá), còn lại giữ trọng số mặc định (1x).

In [2]:
ALL_COLS = CAT_ALL + NUM_ALL
weight_map = {
    "quote_distance": 10, "quote_duration": 10,
    "service_name": 3, "latest_observed_price": 3, "latest_observed_multiplier": 3,
    "history_60m_price_mean": 3, "history_60m_price_std": 3, "history_60m_price_slope_per_minute": 3,
    "latest_observed_quote_distance": 2, "latest_observed_quote_duration": 2,
    "actual_observation_age_minutes": 2, "gio_vn": 2,
}
feature_weights = np.array([weight_map.get(c, 1.0) for c in ALL_COLS])
print("Trong so feature (khac 1.0):")
for cname, w in zip(ALL_COLS, feature_weights):
    if w != 1.0: print(f"  {cname:35} w={w}")

Trong so feature (khac 1.0):
  service_name                        w=3.0
  quote_distance                      w=10.0
  quote_duration                      w=10.0
  actual_observation_age_minutes      w=2.0
  latest_observed_price               w=3.0
  latest_observed_multiplier          w=3.0
  latest_observed_quote_distance      w=2.0
  latest_observed_quote_duration      w=2.0
  history_60m_price_mean              w=3.0
  history_60m_price_std               w=3.0
  history_60m_price_slope_per_minute  w=3.0
  gio_vn                              w=2.0


**3. Huấn luyện 3 cách — theo từng tháng**

In [3]:
def prep(d, cat, num):
    X = d[cat + num].copy()
    for cc in cat: X[cc] = X[cc].astype("category")
    return X

def tao_xgb(fw=None):
    return xgb.XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        tree_method="hist", enable_categorical=True, random_state=42,
        early_stopping_rounds=20, eval_metric="rmse", feature_weights=fw)

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return dict(MAE=mean_absolute_error(y,p), RMSE=mean_squared_error(y,p)**.5,
                R2=r2_score(y,p), MAPE=np.mean(np.abs((y-p)/y))*100)

thangs = sorted(df.evaluation_month.unique())
res = {"A": [], "B": [], "C": []}
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()

    # (A) baseline curated feature (XGBoost, khong weight - de so sanh cong bang thuat toan)
    Xtr_a, Xte_a = prep(tr, CAT_CU, B_NUM_CU), prep(te, CAT_CU, B_NUM_CU)
    mA = tao_xgb().fit(Xtr_a, np.log(tr.base_price), eval_set=[(Xtr_a, np.log(tr.base_price))], verbose=False)
    predA = np.exp(mA.predict(Xte_a))

    # (B) nem het, khong weight
    Xtr_b, Xte_b = prep(tr, CAT_ALL, NUM_ALL), prep(te, CAT_ALL, NUM_ALL)
    mB = tao_xgb().fit(Xtr_b, np.log(tr.base_price), eval_set=[(Xtr_b, np.log(tr.base_price))], verbose=False)
    predB = np.exp(mB.predict(Xte_b))

    # (C) nem het + weight
    mC = tao_xgb(fw=feature_weights).fit(Xtr_b, np.log(tr.base_price), eval_set=[(Xtr_b, np.log(tr.base_price))], verbose=False)
    predC = np.exp(mC.predict(Xte_b))

    for key, pred in [("A", predA), ("B", predB), ("C", predC)]:
        d = pd.DataFrame({"base_price": te.base_price.values, "pred": pred, "evaluation_month": th})
        res[key].append(d)
    print(f"  [{th}] xong ca 3 cach | {time.time()-t0:.1f}s")

  [2026-01] xong ca 3 cach | 253.6s
  [2026-02] xong ca 3 cach | 241.4s
  [2026-03] xong ca 3 cach | 218.0s


**4. So sánh kết quả — tổng gộp & từng test-set nhỏ theo tháng**

In [4]:
all_res = {k: pd.concat(v) for k, v in res.items()}
ten_cach = {"A": "A: Baseline (10 feature)", "B": "B: Nem het (46 feature, khong weight)",
            "C": "C: Nem het + trong so (46 feature)"}
rows = []
for key, d in all_res.items():
    rows.append({"Cach": ten_cach[key], "Test-set": "TAT CA", "n": len(d), **metrics(d.base_price, d.pred)})
    for th in thangs:
        dt = d[d.evaluation_month==th]
        rows.append({"Cach": ten_cach[key], "Test-set": th, "n": len(dt), **metrics(dt.base_price, dt.pred)})
bang = pd.DataFrame(rows).round(2)
print("TONG GOP:"); display(bang[bang["Test-set"]=="TAT CA"])
print("\nTUNG TEST-SET NHO THEO THANG:"); display(bang[bang["Test-set"]!="TAT CA"])

mae_a = bang[(bang["Test-set"]=="TAT CA")&(bang.Cach==ten_cach["A"])].MAE.values[0]
mae_b = bang[(bang["Test-set"]=="TAT CA")&(bang.Cach==ten_cach["B"])].MAE.values[0]
mae_c = bang[(bang["Test-set"]=="TAT CA")&(bang.Cach==ten_cach["C"])].MAE.values[0]
print(f"\nChenh lech B - A (nem het vs baseline)      = {mae_b-mae_a:+,.0f} VND")
print(f"Chenh lech C - A (nem het+weight vs baseline) = {mae_c-mae_a:+,.0f} VND")
print(f"Chenh lech C - B (weight co giup gi khong)    = {mae_c-mae_b:+,.0f} VND")

TONG GOP:


,Cach,Test-set,n,MAE,RMSE,R2,MAPE
0,A: Baseline (10 feature),TAT CA,864360,15043.55,20095.07,0.66,14.6
4,"B: Nem het (46 feature, khong weight)",TAT CA,864360,15049.31,20095.16,0.66,14.6
8,C: Nem het + trong so (46 feature),TAT CA,864360,15048.43,20092.98,0.66,14.6



TUNG TEST-SET NHO THEO THANG:


,Cach,Test-set,n,MAE,RMSE,R2,MAPE
1,A: Baseline (10 feature),2026-01,315360,15074.49,20175.87,0.66,14.61
2,A: Baseline (10 feature),2026-02,234632,14988.45,19960.55,0.66,14.58
3,A: Baseline (10 feature),2026-03,314368,15053.65,20113.91,0.65,14.60
5,"B: Nem het (46 feature, khong weight)",2026-01,315360,15076.29,20162.70,0.66,14.61
6,"B: Nem het (46 feature, khong weight)",2026-02,234632,14990.21,19952.91,0.66,14.59
7,"B: Nem het (46 feature, khong weight)",2026-03,314368,15066.36,20133.06,0.65,14.61
9,C: Nem het + trong so (46 feature),2026-01,315360,15078.93,20162.55,0.66,14.61
10,C: Nem het + trong so (46 feature),2026-02,234632,14985.23,19953.54,0.66,14.58
11,C: Nem het + trong so (46 feature),2026-03,314368,15065.00,20126.75,0.65,14.61



Chenh lech B - A (nem het vs baseline)      = +6 VND
Chenh lech C - A (nem het+weight vs baseline) = +5 VND
Chenh lech C - B (weight co giup gi khong)    = -1 VND


**Kết luận**

- Nếu B và C đều **không thắng rõ** A → 46 feature thêm vào chủ yếu là nhiễu/dư thừa, bộ 10 feature
  hiện tại đã "đủ tốt", ném thêm không giúp gì (khớp với permutation importance đã đo: 92% nằm ở
  quote_distance+quote_duration, phần còn lại gần như 0).
- Nếu B kém hơn A (nhiều feature nhiễu làm model khó học hơn) nhưng C ngang hoặc hơn A → đánh trọng
  số giúp "cứu" được phần nào từ việc ném quá nhiều feature không liên quan.
- Nếu C thắng rõ A → nên mở rộng `_common_train.py` (`B_NUM`) theo hướng này.